In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

here = Path.cwd()
ROOT = next(p for p in [here, *here.parents] if (p / "data" / "supply_chain.db").exists())
con = sqlite3.connect(ROOT / "data" / "supply_chain.db")

def q(sql):
    """Run a SQL query and show the result as a table."""
    return pd.read_sql(sql, con)

q("SELECT * FROM purchase_orders LIMIT 3")

,po_id,supplier_id,category,order_date,promised_date,quantity,unit_cost,po_value,status
0,PO00001,S02,Packaging,2025-01-31,2025-02-12,4400,5.56,24464.0,received
1,PO00002,S06,Ingredients,2025-11-15,2025-12-01,1700,8.56,14552.0,received
2,PO00003,S08,Ingredients,2025-10-25,2025-12-04,2700,6.15,16605.0,received


In [2]:
q("""
SELECT s.name, COUNT(*) AS pos, ROUND(SUM(p.po_value)) AS spend
FROM purchase_orders p
JOIN suppliers s ON s.supplier_id = p.supplier_id
GROUP BY s.name
ORDER BY spend DESC
LIMIT 3
""")

,name,pos,spend
0,Delta Label Co,234,3832653.0
1,Juniper Chemicals,225,3743833.0
2,Apex Polymers,218,3618700.0


In [3]:
q("""
SELECT sh.carrier, COUNT(*) AS shipments,
       ROUND(100.0 * AVG(sh.received_date <= p.promised_date), 1) AS on_time_pct
FROM shipments sh
JOIN purchase_orders p ON p.po_id = sh.po_id
WHERE p.status = 'received'
GROUP BY sh.carrier
ORDER BY on_time_pct
""")

,carrier,shipments,on_time_pct
0,Hapag-Lloyd,271,76.8
1,MSC,279,77.4
2,Maersk,275,81.5
3,Estes,528,88.4
4,XPO,560,88.8
5,UPS Freight,336,92.6


In [4]:
q("""
SELECT CASE WHEN s.country IN ('USA','Canada','Mexico') THEN 'North America' ELSE 'Overseas' END AS lane,
       CASE WHEN strftime('%m', p.order_date) >= '09' THEN 'Sep-Dec' ELSE 'Jan-Aug' END AS period,
       COUNT(*) AS pos,
       ROUND(100.0 * AVG(sh.received_date <= p.promised_date), 1) AS on_time_pct
FROM purchase_orders p
JOIN suppliers s ON s.supplier_id = p.supplier_id
JOIN shipments sh ON sh.po_id = p.po_id
WHERE p.status = 'received'
GROUP BY lane, period
ORDER BY lane, period
""")

,lane,period,pos,on_time_pct
0,North America,Jan-Aug,964,89.4
1,North America,Sep-Dec,460,89.8
2,Overseas,Jan-Aug,630,77.1
3,Overseas,Sep-Dec,195,83.1


In [5]:
q("""
SELECT COUNT(*) AS overdue_pos, ROUND(SUM(po_value)) AS value
FROM purchase_orders
WHERE status = 'open' AND promised_date < '2025-12-31'
""")

,overdue_pos,value
0,10,137224.0
